# 13 · State-clustered standard errors

A referee check on inference: re-estimate the baseline with standard errors
**clustered by state** instead of HC3, since the data have a natural
within-state grouping (and state fixed effects). Clustering only changes the
standard errors, not the coefficient.

**Result:** with 48 state clusters the persistence coefficient is unchanged
(0.0559); its SE rises from 0.014 (HC3) to ~0.020, so p ≈ 0.006 — still
significant at the 1% level. The headline survives clustered inference.

**Manuscript:** robustness / inference check (complements Table 2).

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/cleaned_data_v3.csv")
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "Pop 2010": "pop_2010",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989",
})

cols = ["Index_v2", "income_1989_real_2023", "pct_hs_1990",
        "pop_2010", "firms_2022", "firms_1989"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=cols + ["State"])
df = df[(df["firms_2022"] > 0) & (df["pop_2010"] > 0) & (df["firms_1989"] > 0)]

df["log_firms_2022"] = np.log1p(df["firms_2022"])
df["log_pop_2010"] = np.log(df["pop_2010"])
df[["income_1989_scaled", "pct_hs_1990_scaled"]] = StandardScaler().fit_transform(
    df[["income_1989_real_2023", "pct_hs_1990"]])

formula = ("log_firms_2022 ~ Index_v2 + income_1989_scaled + pct_hs_1990_scaled "
           "+ log_pop_2010 + C(State)")

# same fit, two error structures
hc3 = smf.ols(formula, df).fit(cov_type="HC3")
clustered = smf.ols(formula, df).fit(
    cov_type="cluster", cov_kwds={"groups": df["State"]})

print(f"N = {int(hc3.nobs)}   |   state clusters = {df['State'].nunique()}\n")
comp = pd.DataFrame({
    "coef":      hc3.params,
    "HC3 SE":    hc3.bse,
    "HC3 p":     hc3.pvalues,
    "Cluster SE": clustered.bse,
    "Cluster p":  clustered.pvalues,
}).loc[["Index_v2", "income_1989_scaled", "pct_hs_1990_scaled", "log_pop_2010"]]
print(comp.round(4).to_string())

In [ ]:
# full clustered summary, if you want the complete table
print(clustered.summary())